In [8]:
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, EarlyStoppingCallback
from torchinfo import summary
from datasets import Dataset
from peft import LoraConfig, get_peft_model, PeftModel
import torch
import json, time
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import re
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

In [9]:
BASE = 'pierreguillou/gpt2-small-portuguese'
tok = AutoTokenizer.from_pretrained(BASE)
tok.pad_token = tok.eos_token

modelo = AutoModelForCausalLM.from_pretrained(BASE)

# 🔥 Ajuste fino do LoRA para reduzir overfitting
cfg = LoraConfig(
    r=4,                  # menor rank → menos parâmetros, menos overfitting
    lora_alpha=8,
    target_modules=['c_attn'],  # apenas atenção, como antes
    lora_dropout=0.1,           # mais dropout para regularizar
    task_type='CAUSAL_LM'
)

modelo = get_peft_model(modelo, cfg)
modelo.print_trainable_parameters()

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: pierreguillou/gpt2-small-portuguese
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


trainable params: 147,456 || all params: 124,587,264 || trainable%: 0.1184


In [10]:
def formatar(p):
    return (f"### Pergunta:\n{p['instruction']}\n\n"
            f"### Resposta:\n{p['output']}{tok.eos_token}")

In [11]:
with open('..//saude.jsonl', 'r', encoding='utf-8') as f:
    pares = [json.loads(line) for line in f]

textos = [formatar(p) for p in pares]
tokenizados = tok(textos, truncation=True, max_length=256, padding='max_length')

input_ids = tokenizados['input_ids']
attention_mask = tokenizados['attention_mask']

# Cria labels com padding mascarado (-100)
labels = []
for ids, mask in zip(input_ids, attention_mask):
    lab = ids.copy()
    for i, m in enumerate(mask):
        if m == 0:
            lab[i] = -100
    labels.append(lab)

# Dataset com TODOS os 1200 exemplos (sem split)
dataset = Dataset.from_dict({
    'input_ids': input_ids,
    'attention_mask': attention_mask,
    'labels': labels,
})

# 4. Configurar treino (todas as melhorias, sem validação)
# Cálculo do warmup baseado no total de passos
batch_size = 4
grad_accum = 2
effective_batch_size = batch_size * grad_accum
steps_per_epoch = len(dataset) // effective_batch_size
num_epochs = 8  # você pode manter 8 ou testar 10
total_steps = steps_per_epoch * num_epochs
warmup_steps = int(0.1 * total_steps)

training_args = TrainingArguments(
    output_dir='./resultados_melhorado_sem_val',
    num_train_epochs=num_epochs,
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=grad_accum,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_steps=warmup_steps,
    weight_decay=0.01,
    max_grad_norm=1.0,
    logging_steps=50,
    save_strategy='epoch',
    report_to='none',
)

trainer = Trainer(
    model=modelo,
    args=training_args,
    train_dataset=dataset,
    # sem eval_dataset
)

# 5. Treinar
t0 = time.time()
trainer.train()
print(f'Treinado em {time.time()-t0:.0f}s')

# 6. Salvar o modelo
modelo.save_pretrained('meu_modelo_lora')
tok.save_pretrained('meu_modelo_lora')

Step,Training Loss


KeyboardInterrupt: 

In [ ]:
logs = trainer.state.log_history
df_train = pd.DataFrame([log for log in logs if 'loss' in log])
if 'step' not in df_train.columns:
    df_train['step'] = (df_train.index + 1) * 50

plt.figure(figsize=(10, 5))
sns.lineplot(data=df_train, x='step', y='loss', color='blue')
plt.xlabel('Passos')
plt.ylabel('Perda (Loss)')
plt.title('Curva de Aprendizado - Treino (1200 exemplos)')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:

# 8. Sumário da arquitetura
summary(
    modelo,
    input_data=torch.randint(0, 1000, (1, 256)),
    col_names=["input_size", "output_size", "num_params", "trainable"],
    row_settings=["var_names"],
    depth=3
)

In [ ]:
# Carregar o melhor modelo salvo
base = AutoModelForCausalLM.from_pretrained(BASE)
modelo = PeftModel.from_pretrained(base, 'meu_modelo_lora')
modelo.eval()

In [ ]:
# Função de resposta otimizada (determinística e curta)
def responder(pergunta, penalidade=1.8, n=60):
    p = f"### Pergunta:\n{pergunta}\n\n### Resposta:\n"
    ids = tok(p, return_tensors='pt')
    with torch.no_grad():
        o = modelo.generate(
            **ids,
            max_new_tokens=n,
            do_sample=False,               # modo guloso
            repetition_penalty=penalidade,
            no_repeat_ngram_size=4,
            pad_token_id=tok.eos_token_id
        )
    saida = tok.decode(o[0], skip_special_tokens=True)
    return saida.split('### Resposta:')[-1].strip()

In [ ]:
print(responder('Como posso reduzir o estresse no dia a dia?'))

In [ ]:
print(responder('Quais são os benefícios de uma alimentação saudável?'))

In [ ]:
print(responder('Como posso melhorar minha qualidade de sono?'))

In [ ]:
# ============================================================
# GRÁFICO DE CLUSTERS (t-SNE) - opcional
# ============================================================
with open('..//saude.jsonl', 'r', encoding='utf-8') as f:
    pares = [json.loads(line) for line in f]

instrucoes = [p['instruction'] for p in pares]

def categorizar(texto):
    texto_lower = texto.lower()
    if re.search(r'aliment|dieta|comer|fruta|vegetal|proteína|carboidrato|saudável|receita|nutri', texto_lower):
        return 'Alimentação / Nutrição'
    elif re.search(r'estress|ansiedad|depress|medo|psicológ|mental|emocional|ansiedade|terapia|sono', texto_lower):
        return 'Saúde Mental / Estresse'
    elif re.search(r'exerc|físico|correr|caminhar|atividade|muscul|esporte|alongamento|ioga|yoga', texto_lower):
        return 'Exercício / Atividade Física'
    elif re.search(r'diabetes|câncer|cardia|doença|hipertens|infecção|sintoma|hospital|médico|tratamento|vacina|febre', texto_lower):
        return 'Doenças / Sintomas'
    else:
        return 'Geral / Outros'

categorias = [categorizar(p) for p in instrucoes]

def get_embeddings(texts, model, tokenizer, batch_size=16, max_length=256):
    model.eval()
    all_embeds = []
    device = next(model.parameters()).device
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        inputs = tokenizer(batch_texts, return_tensors='pt',
                           padding=True, truncation=True, max_length=max_length)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)
            last_hidden = outputs.hidden_states[-1]
            mask = inputs['attention_mask'].unsqueeze(-1).float()
            sum_emb = torch.sum(last_hidden * mask, dim=1)
            sum_mask = torch.clamp(mask.sum(dim=1), min=1e-9)
            mean_emb = sum_emb / sum_mask
            all_embeds.append(mean_emb.cpu().numpy())
    return np.vstack(all_embeds)

sample_size = min(500, len(instrucoes))
indices = np.random.choice(len(instrucoes), size=sample_size, replace=False)
sample_texts = [instrucoes[i] for i in indices]
sample_cats = [categorias[i] for i in indices]

print("⏳ Extraindo embeddings...")
embeddings = get_embeddings(sample_texts, modelo, tok, batch_size=16)

print("⏳ PCA + t-SNE...")
pca = PCA(n_components=50, random_state=42)
emb_pca = pca.fit_transform(embeddings)
tsne = TSNE(n_components=2, perplexity=30, n_iter=500, random_state=42)
emb_2d = tsne.fit_transform(emb_pca)

plt.figure(figsize=(12, 10))
cores = {
    'Alimentação / Nutrição': '#2E86C1',
    'Saúde Mental / Estresse': '#E74C3C',
    'Exercício / Atividade Física': '#28B463',
    'Doenças / Sintomas': '#F1C40F',
    'Geral / Outros': '#AEB6BF'
}

for cat in set(sample_cats):
    idx = [i for i, c in enumerate(sample_cats) if c == cat]
    plt.scatter(emb_2d[idx, 0], emb_2d[idx, 1], label=cat, color=cores[cat], alpha=0.7, s=50)

plt.title('Clusters de Perguntas - t-SNE (modelo melhorado)')
plt.xlabel('Dimensão 1')
plt.ylabel('Dimensão 2')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()